[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/01_foundations/05_functions_modules.ipynb)

# 📓 Notebook 5 — Functions and Modules: Building a Reusable Toolkit

> **Module:** Python Fundamentals · **Estimated time:** 35–45 min · **Difficulty:** Beginner

A function is a *reusable block of code with a name*. Whenever you find yourself copy-pasting the same lines, you should wrap them in a function. In AI-driven work this matters more than anywhere else: you call the same API hundreds of times, parse the same response shape, clean the same kinds of input. A well-named function is the difference between a notebook nobody can read and a small library your team can rely on.

In this notebook every example builds a piece of the kind of **toolkit** you would actually keep around: cost calculators, API-response parsers, prompt builders, retry wrappers. By the end you will have written a small but real **data-cleaning + cost-reporting module** you could drop straight into a production project.

## 🎯 Learning objectives

1. Define and call functions with positional, default, and keyword arguments.
2. Use **`*args`** and **`**kwargs`** for flexible signatures.
3. Understand **local vs global** scope and the *mutable default* trap.
4. Write **docstrings** and **type hints** that serve as living documentation.
5. Use **lambdas** in `sorted` / `max` / `filter` calls.
6. **Import** from the standard library and from third-party packages.
7. Compose small functions into a clean pipeline.

## ✅ Prerequisites

Notebooks 1–4.

## 1. Why functions?

Compare these two snippets — both compute the cost of an LLM call from token counts.

```python
# Without a function: works, but you copy-paste this everywhere
in_cost  = tokens_in  / 1000 * 0.0006
out_cost = tokens_out / 1000 * 0.0024
total    = in_cost + out_cost

# With a function: write once, call anywhere
def call_cost(tokens_in, tokens_out, price_in_per_1k, price_out_per_1k):
    return tokens_in  / 1000 * price_in_per_1k + tokens_out / 1000 * price_out_per_1k

total = call_cost(800, 200, 0.0006, 0.0024)
```

The function version has a meaningful **name**, can be **tested** in isolation, can be **reused** anywhere, and can be safely changed in one place. Those four properties are the entire reason for writing functions.

## 2. Defining and calling a function

### 🧠 Mental model — what `def` and a call do to the *call stack*

When you call a function, Python pushes a new **frame** onto the call stack. The frame holds the function's local variables. When the function returns, the frame is popped, and its locals disappear.

```
  def greet(name):              ← function definition
      msg = f"Hello, {name}!"
      return msg

  result = greet("Ada")        ← call

  CALL STACK during the call:

  ┌─────────────────────────┐  ← top of stack
  │  greet's frame          │
  │    name = "Ada"        │   ← parameter (a local variable)
  │    msg  = "Hello, Ada!"│   ← local variable
  └─────────────────────────┘
  ┌─────────────────────────┐  ← module-level frame
  │  __main__               │
  │    result = <pending>   │   ← waiting for greet to return
  └─────────────────────────┘

  AFTER the call (greet's frame is popped):

  ┌─────────────────────────┐
  │  __main__               │
  │    result = "Hello, Ada!"
  └─────────────────────────┘
```

Three consequences worth internalising: (1) **local variables are private** — `msg` doesn't exist outside `greet`. (2) **arguments are local variables** — `name` gets the value of whatever you pass. (3) **`return` ends the frame** — anything after `return` in the same function never runs.


In [ ]:
# The smallest useful function: no arguments, no return value
def log_request():
    print("→ API request sent")

log_request()
log_request()                 # call as often as you like


In [ ]:
# Functions become powerful once they take inputs and return outputs
def call_cost(tokens_in, tokens_out, price_in_per_1k=0.0006, price_out_per_1k=0.0024):
    """USD cost of one LLM call from token usage and per-1K prices."""
    return (tokens_in  / 1000 * price_in_per_1k +
            tokens_out / 1000 * price_out_per_1k)


print(f"Default rates  : ${call_cost(800, 200):.5f}")
print(f"Custom rates   : ${call_cost(800, 200, 0.0010, 0.0040):.5f}")
print(f"Output-heavy   : ${call_cost(120, 1800):.5f}")


### Anatomy of a function definition

```
def   function_name(arg1, arg2=default):
│            │       │       │
│            │       │       └─ default value (optional)
│            │       └───────── parameter list
│            └───────────────── name (snake_case)
└────────────────────────────── the `def` keyword

    """docstring describing what the function does"""
    body
    ...
    return something      # optional; default is None
```

## 3. Parameters with defaults

A parameter with a default value becomes **optional** at the call site. This is the cleanest way to express "the common case is X, but you can override it" — extremely useful for configuration-heavy code.

In [ ]:
def build_prompt(user_question, system="You are a helpful assistant.", style="concise"):
    """Build a chat-style message list for an LLM call."""
    style_hint = {
        "concise":  "Answer in one sentence.",
        "detailed": "Answer in 3–5 sentences.",
        "bullets":  "Reply as a bulleted list.",
    }.get(style, "")
    return [
        {"role": "system", "content": f"{system} {style_hint}".strip()},
        {"role": "user",   "content": user_question},
    ]


# Different call styles — same function
print(build_prompt("How do I cancel my subscription?"))
print()
print(build_prompt("Explain RAG.", style="bullets"))
print()
# Keyword arguments make calls self-documenting on long signatures
print(build_prompt(user_question="Refund timeline?", style="detailed"))


> ⚠️ **Mutable default trap.** Never write `def f(x=[]):`. The list is *shared* across all calls — see the debugging exercise below. Use `def f(x=None): ... if x is None: x = []` instead.

### 🔬 What actually happens — the default object is created *once*, at definition time

Why is the mutable default so dangerous? Because a default value is evaluated **once, when the `def` line runs** — *not* on each call. That one list object is then stored on the function and **reused for every call that doesn't pass its own argument**. So each call quietly appends to the *same* shared list.

```text
   def add(item, bag=[]):        ← the []  is created ONCE, here, and attached to add
                  │
                  ▼
            add.__defaults__ = ( [ <the one shared list> ], )
                  │
   call add("a")  → no bag passed → uses the shared list → ['a']
   call add("b")  → no bag passed → SAME shared list      → ['a', 'b']   😱
```

The cell proves the leak, then shows the standard `None`-sentinel fix:


In [ ]:
# ❌ THE TRAP — one shared default list accumulates across calls
def add(item, bag=[]):
    bag.append(item)
    return bag

# Print each call on its own line so the accumulation is actually visible.
# (Printing all three on one line would mislead: they all return the SAME
#  list object, so by print-time every reference shows the final ['a','b','c'].)
print("buggy call 1:", add("a"))                    # ['a']
print("buggy call 2:", add("b"))                    # ['a', 'b']        — leaked!
print("buggy call 3:", add("c"))                    # ['a', 'b', 'c']   — leaked!
print("the shared default is literally stored on the function:", add.__defaults__)

# ✅ THE FIX — default to None, then create a fresh list inside the body each call
def add_fixed(item, bag=None):
    if bag is None:
        bag = []                                    # NEW list per call → no sharing
    bag.append(item)
    return bag

print("fixed call 1:", add_fixed("a"))              # ['a']
print("fixed call 2:", add_fixed("b"))              # ['b']  — fresh list, no leak
print("fixed call 3:", add_fixed("c"))              # ['c']  — fresh list, no leak


> ⚠️ **Rule of thumb.** Never use a *mutable* object (`[]`, `{}`, `set()`) as a default argument. Use `None` as the sentinel and build the real default inside the function. (This is the exact same root cause as the *mutable class attribute* trap you'll meet in NB 6 — a single mutable object, created once, accidentally shared by everyone.)


## 4. Returning multiple values

A function can return a *tuple* of values; the caller usually unpacks it. For two or three closely related values this is convenient. Once you have four or more *named* outputs, return a **dict** — it's much more readable.

In [ ]:
def usage_summary(records):
    """Return (n_calls, total_tokens, total_cost) for a batch of API records."""
    n = len(records)
    total_tok  = sum(r["tokens_in"] + r["tokens_out"] for r in records)
    total_cost = sum(r["cost_usd"] for r in records)
    return n, total_tok, total_cost


batch = [
    {"tokens_in": 480, "tokens_out": 120, "cost_usd": 0.0006},
    {"tokens_in": 720, "tokens_out": 200, "cost_usd": 0.0009},
    {"tokens_in": 305, "tokens_out":  80, "cost_usd": 0.0004},
]

n, total_tok, total_cost = usage_summary(batch)
print(f"{n} calls, {total_tok} tokens, ${total_cost:.4f} total")


In [ ]:
# When several outputs deserve names, return a dict instead — much clearer
def usage_summary_dict(records):
    """Same as above but returned as a named-field dict."""
    return {
        "n_calls":     len(records),
        "tokens_in":   sum(r["tokens_in"]  for r in records),
        "tokens_out":  sum(r["tokens_out"] for r in records),
        "total_cost":  sum(r["cost_usd"]   for r in records),
    }


import json
print(json.dumps(usage_summary_dict(batch), indent=2))


## 5. Variadic functions — `*args` and `**kwargs`

Sometimes you don't know in advance how many arguments will be passed.

- `*args` collects extra **positional** arguments into a tuple.
- `**kwargs` collects extra **keyword** arguments into a dict.

This is the shape every wrapper / decorator / forwarding-call uses. You see it constantly in libraries.

In [ ]:
def total(*amounts):
    """Sum any number of numeric arguments."""
    return sum(amounts)


print(total(1.20, 2.30, 0.95))
print(total(*[0.20, 0.18, 0.27, 0.31]))   # unpack a list with *


def call_llm_mock(**params):
    """Pretty-print every keyword argument — a stand-in for an SDK call."""
    print("→ calling LLM with parameters:")
    for k, v in params.items():
        print(f"   {k:<12}: {v!r}")


call_llm_mock(model="gpt-4o-mini", temperature=0.0, max_tokens=512, top_p=1.0)


### 🔬 What actually happens — `*`/`**` *pack* on one side and *unpack* on the other

The single `*`/`**` symbol does **opposite jobs** depending on which side of the call it sits on. That symmetry is the whole idea:

```text
  PACKING — in a function DEFINITION (gather many → one)
  ──────────────────────────────────────────────────────
     def f(*args, **kwargs):
            │        │
            │        └── leftover KEYWORD args  →  dict   {'a': 1, 'b': 2}
            └─────────── leftover POSITIONAL args → tuple (1, 2, 3)

  UNPACKING — at the CALL SITE (spread one → many)
  ──────────────────────────────────────────────────────
     f(*[1, 2, 3], **{'a': 1, 'b': 2})
        │            │
        │            └── dict     → spread as keyword args  a=1, b=2
        └─────────────── sequence → spread as positional args 1, 2, 3
```

The cell below shows both directions in one place — packing on the way *in*, unpacking on the way *out*:


In [ ]:
# PACKING: *nums collects positionals into a TUPLE, **opts collects keywords into a DICT
def describe(*nums, **opts):
    print("  nums (a tuple):", nums, "  type:", type(nums).__name__)
    print("  opts (a dict) :", opts, "  type:", type(opts).__name__)

print("Call 1 — pass them one by one:")
describe(1, 2, 3, sep="-", upper=True)


# UNPACKING: * spreads a list into positionals, ** spreads a dict into keywords
print("\nCall 2 — same call, built from a list + dict and unpacked:")
my_numbers = [1, 2, 3]
my_options = {"sep": "-", "upper": True}
describe(*my_numbers, **my_options)        # ← identical to Call 1

# Bonus: * also unpacks directly into other functions — this is sum(*) , print(*), etc.
print("\nUnpacking into print:", end=" ")
print(*my_numbers, sep=" | ")              # print(1, 2, 3, sep=' | ')


> 🧠 **Mental model — `*` is a zipper.** In a *definition* it zips loose arguments **up** into one collection (tuple/dict); at a *call site* it unzips a collection **down** into loose arguments. This is exactly why forwarding wrappers are written `def wrapper(*args, **kwargs): return fn(*args, **kwargs)` — pack everything on the way in, unpack it straight back out to the wrapped function, regardless of its signature. (You'll use precisely this shape in the `retry` and `memoize` decorators in the stretch exercises.)


## 6. Scope — what `x` means depends on *where*

A variable defined **inside** a function is **local**: it exists only during that call.
A variable defined at the top level of a notebook (or module) is **global**.

```
                    ┌────────────────── module / notebook scope ────────────────────┐
                    │  api_budget = 50.00       ←  global                            │
                    │                                                               │
                    │  ┌────────── function scope ────────────┐                     │
                    │  │  spent = 0.0     ← local             │                     │
                    │  └──────────────────────────────────────┘                     │
                    └────────────────────────────────────────────────────────────────┘
```

In [ ]:
api_budget = 50.00              # global

def reset_counter():
    spent = 0.0                  # local — only exists inside this call
    print(f"inside  : spent = {spent}, api_budget = {api_budget}")

reset_counter()
# print(spent)                    # ← would raise NameError — `spent` is gone

print(f"outside : api_budget = {api_budget}")


### 🔬 What actually happens — the LEGB name-resolution rule

When Python hits a bare name like `api_budget` *inside* a function, how does it decide which `api_budget` you mean? It searches **four scopes, in a fixed order**, and stops at the first match. The order spells **LEGB**:

```text
   name used inside a function:  spent
            │
            ▼
   ┌─────────────────────────────────────────────┐
   │  L  Local      → names in THIS function call │  ← looked at first
   ├─────────────────────────────────────────────┤
   │  E  Enclosing  → any outer function wrapping │
   │                  this one (closures)         │
   ├─────────────────────────────────────────────┤
   │  G  Global     → top level of the module /   │
   │                  notebook                    │
   ├─────────────────────────────────────────────┤
   │  B  Built-in   → len, print, sum, range, …   │  ← looked at last
   └─────────────────────────────────────────────┘
            │
            ▼
   first scope that HAS the name wins; if none do → NameError
```

The previous cell already showed two of these rules in action:

- `spent` was found in **L** (Local) — it only existed during the call.
- `api_budget` wasn't in L, so Python fell through to **G** (Global) and found it there.

Two consequences flow from this single rule, and they cause most beginner scope confusion.


### Rule 1 — a *local* name shadows a global of the same name

If a function has its own local `x`, that local **hides** (shadows) any global `x` for the duration of the call. Same word, two completely separate variables — Python picks the Local one first because L comes before G.

```text
  price = 100          (Global price)
       │
       ▼
  def show():
      price = 5        (Local price  ← a NEW, separate variable)
      print(price)     L wins → 5
       │
       ▼
  print(price)         back outside: G → still 100, untouched
```

The cell below proves the global is never touched:


In [ ]:
price = 100                      # GLOBAL price

def show():
    price = 5                    # LOCAL price — a brand-new, separate variable
    print("inside  :", price)    # L wins → 5

show()
print("outside :", price)        # G is untouched → 100

# Same NAME, two different variables. The local one shadowed the global
# only *inside* the call; the global never changed.


### Rule 2 — assigning to a name *anywhere* in a function makes it local

Here is the rule that surprises everyone: **if a function assigns to a name even once, Python treats that name as local for the *entire* function** — including lines *before* the assignment. It decides this when it compiles the function, before any code runs.

So this fails:

```python
counter = 0
def bump():
    counter = counter + 1   # 💥 UnboundLocalError
```

Because `counter =` makes `counter` local, the `counter` on the right-hand side refers to the *local* one — which has no value yet. Python never falls back to the global here; the assignment already settled the verdict.

**The fixes:**

| You want to… | Use |
|---|---|
| Read a global but not rebind it | nothing — just read it (works by LEGB) |
| Rebind a *module-level* (global) name | `global counter` |
| Rebind a name in an *enclosing function* | `nonlocal counter` |

> 🎯 **Intuition.** *Reading* a global is free and automatic. *Rebinding* one (making the name point at a new object) needs an explicit `global` declaration — Python makes you opt in, so you can't reassign module state by accident.


In [ ]:
# 1️⃣ global — rebind a module-level name from inside a function
counter = 0

def bump():
    global counter               # "I really mean the module-level counter"
    counter += 1                 # now this rebinds the global, no error

bump(); bump(); bump()
print("global counter:", counter)        # 3


# 2️⃣ nonlocal — rebind a name owned by the ENCLOSING function (the E in LEGB)
def make_accumulator():
    running = 0                  # enclosing-scope variable
    def add(n):
        nonlocal running         # "the running from make_accumulator, not a new local"
        running += n
        return running
    return add

acc = make_accumulator()
print("nonlocal     :", acc(10), acc(5), acc(100))   # 10 15 115 — state persists in the closure


> 🧠 **Mental model — LEGB is a one-way search, assignment is a declaration.**
> Every bare name is resolved by searching **L → E → G → B** and taking the first hit. But the moment you *assign* to a name inside a function, you've declared it Local for that whole function — and only `global`/`nonlocal` can override that declaration. Read this way, "why is my variable suddenly `UnboundLocalError`?" stops being a mystery: you assigned to it somewhere, so Python made it local.

> ⚠️ **Pitfall — prefer parameters and `return` over `global`.** As the Section-6 best-practice note says, `global` makes functions depend on hidden state and breaks the "inputs in, result out" contract. The accumulator above is the *legitimate* use of enclosed state (a closure). For everything else, pass values in as arguments and hand results back with `return`.


> 🎯 **Best practice.** Pass everything you need *in* through parameters and *return* what you compute. Avoid the `global` keyword — it makes code hard to reason about.

There is one subtle catch with **mutable** objects:

In [ ]:
def add_request(log, request_id):
    log.append(request_id)        # mutates the list IN PLACE
    return log


pending = ["req_001", "req_002"]
add_request(pending, "req_003")
print(pending)                    # ['req_001', 'req_002', 'req_003'] — surprised?


# Lists are passed *by reference*. If you don't want the caller's list to change,
# make a copy inside the function:
def add_request_safe(log, request_id):
    new = log.copy()
    new.append(request_id)
    return new


pending = ["req_001", "req_002"]
new = add_request_safe(pending, "req_003")
print(f"\npending (untouched): {pending}")
print(f"new                 : {new}")


### 🔬 What actually happens — Python is *pass-by-object-reference*

That last result surprises almost everyone, so let's name the mechanism precisely. People often ask "is Python pass-by-value or pass-by-reference?" — and the honest answer is **neither of those labels fits**. Python is **pass-by-object-reference** (sometimes called *call-by-sharing*):

> When you call `f(x)`, the parameter inside `f` is bound to the **very same object** that `x` points to — not a copy of the object, and not a copy of the variable `x`.

```text
   pending ──┐
             ▼
        ┌──────────────────────────┐
        │  list object  ['a','b']  │   ← ONE object in memory
        └──────────────────────────┘
             ▲
   log ──────┘   (inside add_request, the parameter `log`
                  points at the SAME list)
```

`pending` and the parameter `log` are two names for **one** list. So whether the caller sees your change depends entirely on **what you do to that name inside the function:**

| Inside the function you… | Effect on the object | Caller sees it? |
|---|---|---|
| **Mutate** it in place (`log.append(x)`, `log[0]=…`, `log.sort()`) | the shared object changes | ✅ **Yes** |
| **Rebind** the parameter (`log = [...]`) | the name now points at a *new* object | ❌ **No** |

Mutation reaches through the shared reference; rebinding just repoints the local name and leaves the caller's object alone. The two cells below prove each half with `id()`.


In [ ]:
# PROOF A — MUTATING in place changes the caller's list (same object, same id)

def append_in_place(log, item):
    print("   inside  id(log):", id(log))   # SAME id as caller's list
    log.append(item)                        # mutate the shared object

pending = ["req_001", "req_002"]
print("before id(pending):", id(pending))
append_in_place(pending, "req_003")
print("after  pending    :", pending)       # ['req_001', 'req_002', 'req_003'] — changed!
print("after  id(pending):", id(pending))   # SAME id — it's the same object throughout


In [ ]:
# PROOF B — REBINDING the parameter does NOT affect the caller (new object, new id)

def rebind(log, item):
    print("   inside id BEFORE rebind:", id(log))   # same as caller's list
    log = log + [item]                               # `+` builds a NEW list, then rebinds `log`
    print("   inside id AFTER  rebind:", id(log))   # DIFFERENT id — a brand-new object
    print("   inside value           :", log)        # has the item...

pending = ["req_001", "req_002"]
print("before id(pending):", id(pending))
rebind(pending, "req_003")
print("after  pending    :", pending)                # ['req_001', 'req_002'] — UNCHANGED!
print("after  id(pending):", id(pending))            # same id as before — never touched


Read the two ids side by side and the whole concept clicks:

```text
  PROOF A  (mutate)                 PROOF B  (rebind)
  ─────────────────                 ─────────────────
  caller list  id = 140...A         caller list  id = 140...B
  inside  log  id = 140...A   same  inside  log  id = 140...B   same
  log.append(x)  → edits 140...A    log = log+[x] → makes 140...C, NEW
  caller sees the change  ✅         caller still holds 140...B  ❌
```

This ties straight back to **mutability** (NB 3). The *only* reason mutation is visible to the caller is that the object is **mutable** and you changed it in place. Pass an **immutable** object — an `int`, `str`, or `tuple` — and there is no "mutate in place" available at all, so a function can *never* alter the caller's value; the most it can do is rebind its own local name (Proof B), which the caller never sees.


In [ ]:
# Immutables drive the point home: an int can ONLY be rebound, never mutated,
# so a function can never change the caller's int.

def try_to_increment(n):
    n += 1                  # rebinds the LOCAL n to a new int object
    return n               # the only way to communicate a change: return it

x = 10
try_to_increment(x)
print("x after call         :", x)            # 10 — untouched, ints are immutable

x = try_to_increment(x)                        # capture the return value to actually update x
print("x after capturing it :", x)            # 11

# Same story for a tuple — no in-place edit exists:
t = (1, 2)
def try_to_append(seq):
    # seq.append(3)  ← would raise AttributeError: tuples have no .append
    return seq + (3,)      # must build & return a new tuple
print("tuple unchanged      :", t, "→ new:", try_to_append(t))


> 🧠 **Mental model — names are labels, arguments share the object.** Picture every variable as a sticky label on a box (the object). Calling `f(x)` puts a *second* label (the parameter) on the *same box*. **Opening the box and rearranging its contents** (mutation) is visible through every label. **Peeling your label off and sticking it on a different box** (rebinding) only moves *your* label — every other label still points at the original box.

> 💡 **Practical takeaway.** If you don't want a function to alter the caller's list/dict/set, copy it first — exactly what `add_request_safe` did above with `log.copy()` (or `new = log + [item]`, as in Proof B). If you *do* want shared mutation (e.g. accumulating into one list), pass it and mutate in place — but make that intent obvious in the function name and docstring.


## 7. Docstrings — describe what your function does

A **docstring** is a string literal placed as the first statement in a function body. Python, Jupyter, and most editors use it for inline help — and ChatGPT-style coding assistants read it to figure out what your code is supposed to do.

```python
def f(x):
    """Short one-line summary.

    Longer description, parameters, return values, examples.
    """
    ...
```

In [ ]:
def safe_divide(a, b, default=0.0):
    """Divide a by b, returning `default` when b == 0.

    Useful when computing ratios (cost / call, calls / minute, …)
    where the denominator might legitimately be zero on a quiet day.

    Parameters
    ----------
    a, b : numbers
    default : value returned when b == 0 (default 0.0).

    Returns
    -------
    float
    """
    if b == 0:
        return default
    return a / b


# Editors and Jupyter expose this:
help(safe_divide)
print(safe_divide(10, 2))     # 5.0
print(safe_divide(10, 0))     # 0.0 — graceful, not a crash


## 8. Type hints — optional, but very helpful

Python is dynamically typed, but you can **annotate** your signature to document the types you expect:

```python
def call_cost(tokens_in: int, tokens_out: int, price_in_per_1k: float = 0.0006) -> float:
    return tokens_in / 1000 * price_in_per_1k + ...
```

Type hints do not affect runtime behaviour, but they:

- improve editor autocomplete and inline help,
- let tools like `mypy` and Pyright catch type bugs before you run,
- serve as living documentation,
- help AI coding assistants generate better completions.

In [ ]:
from typing import Iterable

def normalise(values: Iterable[float], lower: float = 0.0, upper: float = 1.0) -> list[float]:
    """Linearly rescale `values` into [lower, upper].

    Useful when comparing latencies, satisfaction scores, or costs across
    units — you put each metric on a common 0..1 scale first.
    """
    values = list(values)
    lo, hi = min(values), max(values)
    if hi == lo:
        return [lower] * len(values)
    return [(v - lo) / (hi - lo) * (upper - lower) + lower for v in values]


latencies_ms = [1820, 2150, 1640, 3180, 2410, 1740]
print(normalise(latencies_ms))


## 9. Lambdas — tiny one-line functions

A **lambda** is an anonymous function. Use it where you would otherwise have to define a one-line `def` just to pass it somewhere — sorting, mapping, filtering. You saw this in NB 3 already: `sorted(records, key=lambda r: r["cost"])`.

In [ ]:
# Sort API calls by cost descending — find the priciest offenders
calls = [
    {"id": "req_001", "cost": 0.0017},
    {"id": "req_002", "cost": 0.0043},
    {"id": "req_003", "cost": 0.0009},
    {"id": "req_004", "cost": 0.0062},
    {"id": "req_005", "cost": 0.0021},
]

ranked = sorted(calls, key=lambda c: c["cost"], reverse=True)
for c in ranked:
    print(f"  {c['id']}  ${c['cost']:.4f}")


> 💡 **Don't overuse lambdas.** For anything longer than one expression, give it a name with `def` — your future self reading the code will thank you.

## 10. Importing — using other people's code

You will write these three lines thousands of times:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
```

Four flavours of import to recognise:

| Style                                | Use when                                            |
|--------------------------------------|-----------------------------------------------------|
| `import math`                        | you want the whole module under its name (`math.sqrt`) |
| `import numpy as np`                 | you want an alias (community standard: `np`, `pd`, `plt`) |
| `from collections import Counter`    | you only need one symbol — keeps code uncluttered    |
| `from x import *`                    | ⚠️ Almost never. Pollutes the namespace, hard to debug. |

In [ ]:
# Standard library — ships with Python, never needs `pip install`
import math
import random
import statistics
import json

print(f"sqrt(2)  = {math.sqrt(2):.4f}")
print(f"pi       = {math.pi:.6f}")

random.seed(0)
print(f"random   = {random.choice(['☀️', '🌧️', '❄️'])}")

print(f"median   = {statistics.median([1, 2, 3, 4, 100])}  ← robust to outliers")
print(f"json     = {json.dumps({'a': 1})}")


In [ ]:
# The data-science 'Big 3' — we'll use them heavily in NB 7 (pandas),
# NB 8 (NumPy), and NB 9 (visualization — via pandas .plot() and seaborn)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"numpy      version: {np.__version__}")
print(f"pandas     version: {pd.__version__}")
print(f"matplotlib version: {plt.matplotlib.__version__}")


## 11. Putting it together — a tiny data-cleaning + cost-reporting toolkit

Four small functions composed into a pipeline. Each does **one** thing well — that's the modular mindset that makes large codebases manageable.

In [ ]:
from typing import Iterable, Optional

# ----- 1. Parsing -----
def parse_number(text: str) -> Optional[float]:
    """Try to convert a string to float; return None when it cannot."""
    try:
        return float(text)
    except (ValueError, TypeError):
        return None


# ----- 2. Cleaning -----
def clean_latencies(values: Iterable, min_ms: float = 0.0, max_ms: float = 60_000.0) -> list[float]:
    """Drop non-numeric and out-of-range latency readings."""
    out = []
    for v in values:
        n = parse_number(v) if isinstance(v, str) else v
        if isinstance(n, (int, float)) and min_ms <= n <= max_ms:
            out.append(float(n))
    return out


# ----- 3. Costing -----
def call_cost(tokens_in: int, tokens_out: int,
              price_in_per_1k: float = 0.0006,
              price_out_per_1k: float = 0.0024) -> float:
    """USD cost of one LLM call."""
    return tokens_in / 1000 * price_in_per_1k + tokens_out / 1000 * price_out_per_1k


# ----- 4. Summarising -----
def report(records: list[dict]) -> dict:
    """One-line numeric summary across a batch of API records."""
    if not records:
        return {"n": 0, "total_cost": 0.0, "mean_latency": None}
    total_cost = sum(r["cost"] for r in records)
    clean      = clean_latencies(r["latency_ms"] for r in records)
    return {
        "n":            len(records),
        "total_cost":   round(total_cost, 4),
        "mean_latency": round(sum(clean) / len(clean), 1) if clean else None,
    }


# Use them together: parse raw → clean → cost → report
raw_batch = [
    {"tokens_in": 480, "tokens_out": 120, "latency_ms": "1820"},
    {"tokens_in": 720, "tokens_out": 200, "latency_ms": "3180"},
    {"tokens_in": 305, "tokens_out":  80, "latency_ms": "n/a"},      # bad row
    {"tokens_in": 610, "tokens_out": 175, "latency_ms": "2110"},
]

# Compute cost for each record
for r in raw_batch:
    r["cost"] = call_cost(r["tokens_in"], r["tokens_out"])

print(report(raw_batch))


**Notice what we did.** Each function has one job (parse / clean / cost / summarise) and a docstring. We can swap any one of them later — say, switch to a different pricing model — without touching the others. That is the *modular* mindset that makes a 50-line analysis grow into a 5,000-line system without becoming unreadable.

## 🧪 Practice exercises

### Exercise 1 — ⭐ Two unit converters

Write `tokens_to_usd(tokens, price_per_1k)` and `usd_to_tokens(usd, price_per_1k)` with docstrings. Verify them on a few values:

- 1000 tokens at $0.0006/1K → $0.0006
- $0.10 budget at $0.0006/1K → 166,666 tokens

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def tokens_to_usd(tokens: int, price_per_1k: float) -> float:
    """Convert a token count to USD cost given a price per 1K tokens."""
    return tokens / 1000 * price_per_1k

def usd_to_tokens(usd: float, price_per_1k: float) -> int:
    """Convert a USD budget to the number of tokens it buys."""
    return int(usd / price_per_1k * 1000)

# Sanity checks
print(f"1000 tokens   → ${tokens_to_usd(1000, 0.0006):.4f}")
print(f"$0.10 budget  → {usd_to_tokens(0.10, 0.0006):,} tokens")

# Round-trip
tok = 250_000
usd = tokens_to_usd(tok, 0.0009)
back = usd_to_tokens(usd, 0.0009)
print(f"Round-trip: {tok} → ${usd:.4f} → {back} tokens")
```
</details>

### Exercise 2 — ⭐⭐ Stats function

Write `stats(values)` returning a dict with `n`, `mean`, `median`, `std`. Implement it yourself with only `sum`, `len`, `sorted`, and arithmetic — no `statistics` or `numpy`. Test it on a list of latencies.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def stats(values):
    """Return n / mean / median / std for a list of numbers."""
    n = len(values)
    if n == 0:
        return {"n": 0, "mean": None, "median": None, "std": None}
    mean = sum(values) / n
    s = sorted(values)
    if n % 2 == 1:
        median = s[n // 2]
    else:
        median = (s[n // 2 - 1] + s[n // 2]) / 2
    var = sum((x - mean) ** 2 for x in values) / n
    return {"n": n, "mean": mean, "median": median, "std": var ** 0.5}


latencies = [1820, 2150, 1640, 3180, 2410, 1740, 2530]
print(stats(latencies))
```

This is exactly what `numpy` will do faster and shorter (`np.mean`, `np.median`, `np.std`) once we get to NB 8 — but writing it once by hand teaches you what those calls are actually doing.
</details>

### Exercise 3 — ⭐⭐ Higher-order function

Write `keep_if(values, predicate)` that returns only items where `predicate(item)` is true. Test it with two different lambdas — for instance "keep slow requests (latency > 2500ms)" and "keep cheap requests (cost < $0.001)".

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def keep_if(values, predicate):
    """Filter a list with a predicate function."""
    return [v for v in values if predicate(v)]


calls = [
    {"id": "req_001", "latency_ms": 1820, "cost": 0.0017},
    {"id": "req_002", "latency_ms": 3180, "cost": 0.0043},
    {"id": "req_003", "latency_ms": 1640, "cost": 0.0009},
    {"id": "req_004", "latency_ms": 2840, "cost": 0.0062},
    {"id": "req_005", "latency_ms": 2150, "cost": 0.0021},
]

slow  = keep_if(calls, lambda c: c["latency_ms"] > 2500)
cheap = keep_if(calls, lambda c: c["cost"] < 0.001)

print("slow :", [c["id"] for c in slow])
print("cheap:", [c["id"] for c in cheap])
```

Passing a function as an argument is called a **higher-order function**. The built-ins `filter`, `map`, `sorted` all work this way — and you'll use the same trick when scoring or ranking AI outputs.
</details>

### Exercise 4 — ⭐⭐ Default-argument pitfall

What does this print after the third call? Predict, then run.

In [ ]:
def add(item, bag=[]):
    bag.append(item)
    return bag

print(add("a"))
print(add("b"))
print(add("c"))


<details>
<summary>💡 <b>Solution & fix</b></summary>

Output is `['a']`, `['a', 'b']`, `['a', 'b', 'c']` — the default list is **shared across all calls**. The list object is created *once*, when the function is defined, not once per call.

Fix:

```python
def add(item, bag=None):
    if bag is None:
        bag = []
    bag.append(item)
    return bag
```

The `None` sentinel + create-inside-the-function is the standard Python idiom. This bug is so common it has its own page in every Python style guide.
</details>

### Exercise 5 — ⭐⭐ Debug me 🐞

Find the bug — `mean_cost` should return the average cost of a batch but is consistently off.

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
def mean_cost(records):
    total = 0
    for r in records:
        total = r["cost"]        # bug!
    return total / len(records)


calls = [
    {"id": "req_001", "cost": 0.0017},
    {"id": "req_002", "cost": 0.0043},
    {"id": "req_003", "cost": 0.0009},
]
print(f"Mean cost: ${mean_cost(calls):.5f}   (expected ~$0.00230)")


<details>
<summary>💡 <b>Solution</b></summary>

`total = r["cost"]` *overwrites* the running total each iteration. The right operator is `+=`:

```python
def mean_cost(records):
    if not records:
        return None
    total = 0
    for r in records:
        total += r["cost"]
    return total / len(records)
```

Or, one-liner with a generator:

```python
def mean_cost(records):
    return sum(r["cost"] for r in records) / len(records) if records else None
```

This is the **single most common accumulator bug** — you'll see it again in NB 8 with NumPy and again in NB 14 with model training loops. Always check whether you mean `=` or `+=`.
</details>

## 🧠 Stretch exercises

Four more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — ⭐⭐⭐ A `retry` decorator

Write a decorator `retry(max_attempts=3, delay=0.1)` that retries a function on any exception. Apply it to a flaky function that succeeds only on the 3rd try, and confirm it returns the right value.


In [ ]:
# Your code here  👇
import time
import functools


<details>
<summary>💡 <b>Solution</b></summary>

```python
import time, functools

def retry(max_attempts=3, delay=0.1):
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        raise
                    print(f"  attempt {attempt} failed: {e!r} — retrying")
                    time.sleep(delay)
        return wrapper
    return deco


@retry(max_attempts=4, delay=0.01)
def flaky(state=[0]):
    state[0] += 1
    if state[0] < 3:
        raise RuntimeError("not yet")
    return f"OK on attempt {state[0]}"

print(flaky())
```

**Why decorators matter.** They let you *layer* concerns (retry,
logging, timing, caching) without touching the wrapped function.
The same shape you'll see for `@app.route` in Flask, `@task` in
Celery, `@pytest.fixture` in tests.

</details>

### Stretch exercise B — ⭐⭐⭐ Compose a 3-step pipeline

Write three small functions — `strip_whitespace(s)`, `lowercase(s)`, `keep_alphanum(s)` — and a `compose(*fns)` helper that returns a function chaining them in order. Apply it to `"  Hello, World 2024!  "` and confirm the result.


In [ ]:
# Your code here  👇
text = "  Hello, World 2024!  "


<details>
<summary>💡 <b>Solution</b></summary>

```python
def strip_whitespace(s: str) -> str: return s.strip()
def lowercase(s: str) -> str:        return s.lower()
def keep_alphanum(s: str) -> str:    return "".join(c for c in s if c.isalnum() or c == " ")

def compose(*fns):
    """Return a function that applies each fn in order, left to right."""
    def composed(x):
        for fn in fns:
            x = fn(x)
        return x
    return composed


pipeline = compose(strip_whitespace, lowercase, keep_alphanum)
print(repr(pipeline("  Hello, World 2024!  ")))    # → 'hello world 2024'
```

**Why composition is the right shape for text cleaners.** Each step
is independently testable, reusable, and you can drop one in or out
without surgery on the others. This is exactly the pattern
`sklearn.pipeline.Pipeline` codifies for ML transformations.

</details>

### Stretch exercise C — ⭐⭐⭐ Memoizing decorator

Write a decorator `memoize` that caches function results so repeated calls with the same arguments return instantly. Test it against a Fibonacci function — without memoization, `fib(30)` takes noticeably long; with memoization, it's instant.

Assume positional integer arguments only (keep the cache key simple).

In [ ]:
# Your code here  👇
def memoize(fn):
    ...

@memoize
def fib(n):
    return n if n < 2 else fib(n-1) + fib(n-2)

# print(fib(30))


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def memoize(fn):
    cache = {}
    def wrapper(*args):
        if args not in cache:
            cache[args] = fn(*args)
        return cache[args]
    return wrapper

@memoize
def fib(n):
    return n if n < 2 else fib(n-1) + fib(n-2)

print(fib(30))   # 832040, instant
```

**Reasoning.** The decorator wraps `fn` in a new function that consults a closure-local `cache` dict before delegating. Two important details. (1) The cache key is `args` (the positional-arg tuple) — tuples are hashable, so they can be dict keys. If you wanted to support keyword args, you'd have to freeze them too: `cache_key = args + tuple(sorted(kwargs.items()))`. (2) In real code, you should use the battle-tested `functools.lru_cache` (or `cache`) decorator from the standard library — it's a single line `@lru_cache(maxsize=None)` above the function. Writing your own is great for understanding what happens under the hood.
</details>

### Stretch exercise D — ⭐⭐⭐ Flexible greeter with `*args` and `**kwargs`

Write a single function `greet(*names, greeting="Hello", emoji="👋")` that prints one greeting per name. Note that `greeting` and `emoji` sit *after* `*names`, which makes them **keyword-only** — callers must spell them out by name. For example:

```python
greet("Ada", "Linus", greeting="Hi", emoji="🚀")
```
should print:
```
Hi, Ada 🚀
Hi, Linus 🚀
```

Calling `greet()` with no names should print exactly `Nobody to greet.`.

In [ ]:
# Your code here  👇
def greet(*names, greeting="Hello", emoji="👋"):
    ...

greet("Ada", "Linus", greeting="Hi", emoji="🚀")


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def greet(*names, greeting="Hello", emoji="👋"):
    if not names:
        print("Nobody to greet.")
        return
    for n in names:
        print(f"{greeting}, {n} {emoji}")

greet("Ada", "Linus", greeting="Hi", emoji="🚀")
greet()
```

**Reasoning.** Three useful patterns at once. (1) `*names` collects all positional arguments into a tuple, so `greet("Ada", "Linus")` works without a list. (2) Keyword-only arguments with defaults give callers a self-documenting interface — `greeting="Hi"` reads better at the call site than `greet("Ada", "Hi")`. (3) The empty-input guard at the top is a habit worth building — handle the boring edge case once at the entrance and the rest of the function can assume `names` is non-empty.
</details>

## 🎁 Bonus mini-project — A modular feedback-cost reporter

Build three small functions:

1. `classify(text)` returns `"complaint" / "praise" / "question"` based on simple keyword rules.
2. `process(item)` takes `{"id": ..., "text": ..., "tokens_in": ..., "tokens_out": ...}` and returns a dict with `id`, `label`, `cost`.
3. `report(items)` prints a tidy table plus the totals row (count, total cost).

Use docstrings throughout, and reuse `call_cost` from earlier.

In [ ]:
# Your code here  👇
feedback = [
    {"id": "F-001", "text": "Love the new dashboard, it's amazing!",      "tokens_in": 480, "tokens_out": 120},
    {"id": "F-002", "text": "How do I cancel my subscription?",            "tokens_in": 410, "tokens_out": 145},
    {"id": "F-003", "text": "Refund needed — wrong charge",                "tokens_in": 360, "tokens_out": 110},
    {"id": "F-004", "text": "Thanks, customer support was fantastic.",     "tokens_in": 320, "tokens_out":  95},
    {"id": "F-005", "text": "Bug: the app crashes when uploading.",        "tokens_in": 540, "tokens_out": 180},
]


<details>
<summary>💡 <b>Solution</b></summary>

```python
def classify(text: str) -> str:
    """Tiny keyword-based classifier for feedback."""
    t = text.lower()
    if any(w in t for w in ("refund", "bug", "broken", "crash", "wrong")):
        return "complaint"
    if any(w in t for w in ("love", "amazing", "thanks", "great", "fantastic")):
        return "praise"
    if "?" in text or any(t.startswith(w) for w in ("how", "why", "when", "what", "can")):
        return "question"
    return "other"


def process(item: dict) -> dict:
    """Compute label and cost for one feedback item."""
    return {
        "id":    item["id"],
        "label": classify(item["text"]),
        "cost":  round(call_cost(item["tokens_in"], item["tokens_out"]), 5),
    }


def report(items: list[dict]) -> None:
    """Print a clean per-item table plus totals."""
    processed = [process(i) for i in items]

    print(f"{'ID':<6}  {'label':<10}  cost ($)")
    print("-" * 30)
    for r in processed:
        print(f"{r['id']:<6}  {r['label']:<10}  {r['cost']:.5f}")
    total_cost = sum(r["cost"] for r in processed)
    print("-" * 30)
    print(f"{'Total':<6}  {len(processed):<10}  {total_cost:.5f}")


report(feedback)
```

**Why this matters.** This is structurally the same as any real customer-feedback pipeline:

1. *classify* assigns a label,
2. *process* enriches each record with the derived fields,
3. *report* presents the result.

When you eventually replace `classify` with an LLM call (NB 21), the rest of the pipeline doesn't change at all. That decoupling is what good function design buys you.
</details>

## 🧠 Key takeaways

1. A **function** bundles reusable logic behind a meaningful name.
2. Parameters can have **defaults** and be passed by **position** or **keyword**.
3. **`*args`** and **`**kwargs`** collect extra positional and keyword arguments.
4. Return one value, a tuple, or a **dict** when several outputs deserve names.
5. Variables inside a function are **local**; passing mutable objects gives the function the power to mutate them.
6. Document with **docstrings** and **type hints** — they pay off the first time a colleague (or you, six months later) opens the file.
7. **Compose small functions** — one job each — into a pipeline.
8. Imports give you the giants' shoulders: standard library + numpy/pandas/matplotlib + scikit-learn.

## ✅ Self-assessment

- [ ] Define a function with positional, default, and keyword arguments
- [ ] Use `*args` and `**kwargs` to write flexible signatures
- [ ] Write a docstring and add type hints
- [ ] Pass a function as an argument (or use a lambda for a one-liner)
- [ ] Explain the **mutable default trap** and the fix
- [ ] Import a third-party module under an alias
- [ ] Compose three small functions into a working pipeline

## 🚀 Next step

Continue with **Notebook 6 — Classes and Object-Oriented Programming**, where you'll bundle data *and* behaviour into a single reusable type — the last core Python skill before you start pulling in real-world data.